In [ ]:
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 自动获取 data 文件夹下所有的 csv 文件
csv_files = glob.glob('data/*_history.csv')

all_data = []

# 2. 循环读取每个战甲的数据
for file in csv_files:
    # 从文件名中提取战甲名称 (例如: data\wisp_prime_set_history.csv -> wisp_prime_set)
    wf_name = os.path.basename(file).replace('_history.csv', '').replace('_', ' ').title()
    
    try:
        df = pd.read_csv(file)
        if not df.empty:
            # 确保日期列是时间格式
            df['datetime'] = pd.to_datetime(df['datetime']).dt.date
            # 记录这个战甲的名字
            df['Warframe'] = wf_name 
            all_data.append(df)
    except Exception as e:
        print(f"读取 {wf_name} 失败: {e}")

# 3. 把所有战甲的数据合并成一个巨大的 DataFrame
if all_data:
    master_df = pd.concat(all_data, ignore_index=True)
    print(f"成功加载了 {len(csv_files)} 个战甲的数据！总记录数: {len(master_df)}")
else:
    print("没有找到任何有效数据，请检查 data/ 文件夹是否为空，或者 GitHub Action 是否成功运行过。")
    

In [ ]:
# 计算每个战甲的平均价格和价格波动（标准差）
stats = master_df.groupby('Warframe')['avg_price'].agg(['mean', 'std', 'min', 'max']).reset_index()
# 计算波动率 (标准差 / 均值)
stats['volatility_rate'] = stats['std'] / stats['mean'] 

# 排序找出波动最大的前 5 个
volatile_top5 = stats.sort_values(by='volatility_rate', ascending=False).head(5)

# 画图
plt.figure(figsize=(10, 5))
sns.barplot(data=volatile_top5, x='volatility_rate', y='Warframe', palette='coolwarm')
plt.title('Top 5 Most Volatile Warframe Platinum Prices')
plt.xlabel('Volatility Rate (Standard Deviation / Mean)')
plt.ylabel('Warframe Set')
plt.show()